# Волна 2: соответствие "прогноз -> целое", выученное по фактам

База - лучший конфиг на сегодня (`05_combo`): v5 + LightGBM, порог 0.55 для недель 1-4 и 0.5 для недель 5-9, целое под SMAPE.

Целое под SMAPE - самый большой наш шаг (v4: -0.004...-0.005). Сейчас целое выбирается по Пуассону вокруг прогноза регрессора q. Два направления от Пуассона мы проверили - больше штук (`amount_scale`) и ближе к медиане (отрицательное биномиальное) - оба хуже. Но фактическое распределение продаж при данном q может отличаться от Пуассона неравномерно. Здесь соответствие учится прямо по фактам (`pipeline.learn_integer_map`): для корзины прогноза - целое с наименьшим средним SMAPE по неделям с продажей; мало недель в корзине - как Пуассон; целое не убывает с прогнозом.

**Честная проверка:** соответствие учим на двух фолдах и применяем к третьему, по кругу. Для справки - выученное на всех трёх (оптимистично: учили на том же). Проходит, если на отложенных фолдах не хуже базы на обоих сезонных, а на `recent` хуже не больше чем на 0.005. Если прошло - соответствие по всем трём фолдам сохраняется в `intmap_05_intmap.json` для финала.

In [1]:
import sys; sys.path[:0] = ["../src", "../../src"]   # код проекта - в src/ (ноутбук в notebooks/ или notebooks/05_experiments/)

import json
import numpy as np
import pandas as pd
import core
from core import *
import classifier, regressor, pipeline
from pipeline import decide, learn_integer_map, member_settings, member_weights

NAME = "05_intmap"
if ON_KAGGLE:   # как валидировались v5 и сборка: каждая 2-я неделя отсчёта
    classifier.ORIGIN_STEP = regressor.ORIGIN_STEP = 2
BEST = {"threshold": (0.55, 0.5), "integer": "smape", "ensemble": [{}, {"random_seed": 7}, {"library": "lightgbm"}]}
SEASON = ["season2025", "season2024"]
EDGES = list(np.round(np.r_[np.arange(1, 3, 0.1), np.arange(3, 6, 0.25), np.arange(6, 10.5, 0.5), np.arange(11, 21, 1)], 2))
sales, stock = load_data()
len(EDGES) - 1, "корзин прогноза"

(50, 'корзин прогноза')

## Модели лучшего конфига по фолдам

Вероятность продажи p и количество q (пары × недели), как в `pipeline`; факт y.

In [2]:
DATA = {}
members = BEST["ensemble"]
for fold, cut in FOLDS.items():
    pairs, s_cut, st_cut, truth, base = core._fold_data(fold, cut)
    weeks = forecast_weeks(s_cut.date.max())
    p = np.average([classifier.fit_predict(s_cut, st_cut, pairs, weeks, **member_settings(m, "clf"))["p"] for m in members],
                   axis=0, weights=member_weights(members, "clf"))
    q = np.expm1(np.average([np.log1p(regressor.fit_predict(s_cut, st_cut, pairs, weeks, **member_settings(m, "reg"))["amount"])
                             for m in members], axis=0, weights=member_weights(members, "reg")))
    y = truth.quantity.to_numpy().reshape(len(weeks), len(pairs)).T
    DATA[fold] = dict(pairs=pairs, weeks=weeks, truth=truth, base=base, p=p, q=q, y=y)
    print(fold, "готов", flush=True)

season2025 готов


[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN
[LightGBM] [Warning] Met negative value in categorical features, will convert it to NaN


season2024 готов


recent готов


## Соответствие: учим на двух фолдах, проверяем на третьем

In [3]:
def score_of(fold, integer):
    d = DATA[fold]
    values = decide(d["p"], d["q"], threshold=BEST["threshold"], integer=integer)
    pred = to_long(pd.DataFrame(values, index=pd.MultiIndex.from_frame(d["pairs"]), columns=d["weeks"]))
    m = compute_metrics(d["truth"], pred)
    return {**m, "score": score_minmax(m, d["base"]), "прогноз / факт": pred.quantity.sum() / d["truth"].quantity.sum()}


def learned_on(folds):
    return learn_integer_map(np.concatenate([DATA[f]["q"].ravel() for f in folds]),
                             np.concatenate([DATA[f]["y"].ravel() for f in folds]), EDGES)


rows = []
for fold in FOLDS:
    others = [f for f in FOLDS if f != fold]
    rows.append({"вариант": "лучший конфиг (Пуассон)", "фолд": fold, **score_of(fold, "smape")})
    rows.append({"вариант": "выучено на двух других фолдах", "фолд": fold, **score_of(fold, learned_on(others))})
    rows.append({"вариант": "выучено на всех трёх (справочно)", "фолд": fold, **score_of(fold, learned_on(list(FOLDS)))})
grid = pd.DataFrame(rows)
grid.to_csv(OUT_DIR / f"grid_{NAME}.csv", index=False)
grid.pivot_table(index="вариант", columns="фолд", values=["score", "smape", "wape", "bias"]).round(4)

bias                         score  \
фолд                               recent season2024 season2025  recent   
вариант                                                                   
выучено на всех трёх (справочно)  75.9617    86.0129    80.8604  0.7814   
выучено на двух других фолдах     76.9402    84.9119    79.6300  0.7845   
лучший конфиг (Пуассон)           74.7154    85.0521    80.1781  0.7831   

                                                          smape             \
фолд                             season2024 season2025   recent season2024   
вариант                                                                      
выучено на всех трёх (справочно)     0.8170     0.8045  58.3023    42.7618   
выучено на двух других фолдах        0.8164     0.8030  58.3800    42.8487   
лучший конфиг (Пуассон)              0.8195     0.8078  58.4244    42.8649   

                                                wape                        
фолд                             season2025   recent season2024 season2025  
вариант                                                                     
выучено на всех трёх (справочно)    57.2731  46.8889    62.5684    50.8860  
выучено на двух других фолдах       57.3159  47.2402    62.4210    50.7123  
лучший конфиг (Пуассон)             57.5017  47.0000    62.6191    51.1018

## Выбор и соответствие для финала

In [4]:
scores = grid.pivot_table(index="вариант", columns="фолд", values="score")[list(FOLDS)]
BASE = scores.loc["лучший конфиг (Пуассон)"]
cv = scores.loc["выучено на двух других фолдах"]
PASSED = bool((cv[SEASON] <= BASE[SEASON]).all() and cv["recent"] <= BASE["recent"] + 0.005)
print("Пуассон:            ", BASE.round(4).to_dict())
print("выучено (отложенно):", cv.round(4).to_dict(), "| разница:", (cv - BASE).round(4).to_dict())
print("проходит:", PASSED)
FINAL_MAP = learned_on(list(FOLDS))
(OUT_DIR / f"intmap_{NAME}.json").write_text(json.dumps(FINAL_MAP))
from pipeline import smape_choice
mids = [(lo + hi) / 2 for lo, hi in zip(EDGES[:-1], EDGES[1:])]
pd.DataFrame({"прогноз от": EDGES[:-1], "до": EDGES[1:], "целое (по фактам)": FINAL_MAP["k"],
              "целое (Пуассон)": [smape_choice(np.array([[m]]))[0][0, 0] for m in mids]})

Пуассон:             {'season2025': 0.8078, 'season2024': 0.8195, 'recent': 0.7831}
выучено (отложенно): {'season2025': 0.803, 'season2024': 0.8164, 'recent': 0.7845} | разница: {'season2025': -0.0048, 'season2024': -0.0032, 'recent': 0.0014}
проходит: True


,прогноз от,до,целое (по фактам),целое (Пуассон)
0,1.00,1.10,1.0,1.0
1,1.10,1.20,1.0,1.0
2,1.20,1.30,1.0,1.0
3,1.30,1.40,1.0,1.0
4,1.40,1.50,1.0,1.0
5,1.50,1.60,1.0,1.0
6,1.60,1.70,1.0,1.0
7,1.70,1.80,2.0,2.0
8,1.80,1.90,2.0,2.0
9,1.90,2.00,2.0,2.0


## Что в итоге

В финал соответствие не вошло. На сезонных фолдах оно лучше Пуассона, но на `recent`, самом близком к тесту по времени, завышает объём (прогноз / факт 0.90 против 0.83) и ухудшает WAPE, Bias и RMSLE. В финале целое под SMAPE по Пуассону (`src/final.py`), само выученное соответствие сохранено в `results/integer_map.json`.